# Cost calculator for LLM router

In [1]:
PRICING = {
    "gpt-4": {
        "input": 30.00,   # $30 per 1M input tokens
        "output": 60.00   # $60 per 1M output tokens
    },
    "gpt-3.5-turbo": {
        "input": 0.50,
        "output": 1.50
    },
    "llama-3.1-8b": {  # Groq
        "input": 0.05,
        "output": 0.08
    },
    "mixtral-8x7b": {  # Groq
        "input": 0.24,
        "output": 0.24
    }
}

In [3]:
def calculate_cost(model: str ,input_tokens: int, output_tokens: int)-> dict:
    pricing = PRICING[model]

    input_cost = (input_tokens/1_000_000) * pricing["input"]
    output_cost = (output_tokens/1_000_000) * pricing["output"]
    total_cost = input_cost + output_cost

    return {
        "model": model,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "input_cost_usd": round(input_cost, 6),
        "output_cost_usd": round(output_cost, 6),
        "total_cost_usd": round(total_cost, 6)
    }

In [5]:
for model in PRICING:
    result = calculate_cost(model, input_tokens=216, output_tokens=100)
    print(f"{model}: ${result['total_cost_usd']}")

gpt-4: $0.01248
gpt-3.5-turbo: $0.000258
llama-3.1-8b: $1.9e-05
mixtral-8x7b: $7.6e-05


# routing cost simulator

In [8]:
def simulate_routing_cost(queries: list, routing_strategy: str) -> dict:
    total_cost = 0
    model_usage = {model: 0 for model in PRICING}
    
    for query in queries:
        complexity = query["complexity"]  # "simple", "medium", "complex"
        input_tokens = query["input_tokens"]
        output_tokens = query["output_tokens"]
        
        if routing_strategy == "always_gpt4":
            model = "gpt-4"
            
        elif routing_strategy == "always_llama":
            model = "llama-3.1-8b"
            
        elif routing_strategy == "smart_routing":
            # WHY this mapping: balance cost vs capability
            if complexity == "simple":
                model = "llama-3.1-8b"
            elif complexity == "medium":
                model = "mixtral-8x7b"
            else:  # complex
                model = "gpt-4"
        
        cost = calculate_cost(model, input_tokens, output_tokens)
        total_cost += cost["total_cost_usd"]
        model_usage[model] += 1
    
    return {
        "strategy": routing_strategy,
        "total_cost_usd": round(total_cost, 6),
        "model_usage": {k: v for k, v in model_usage.items() if v > 0},
        "avg_cost_per_query": round(total_cost / len(queries), 6)
    }


In [9]:
import random
random.seed(42)

queries = []
for _ in range(1000):
    complexity = random.choices(
        ["simple", "medium", "complex"],
        weights=[0.6, 0.3, 0.1]  # 60% simple, 30% medium, 10% complex
    )[0]
    
    # Token counts vary by complexity
    # WHY: complex queries need more context and produce longer answers
    if complexity == "simple":
        input_tokens = random.randint(150, 250)
        output_tokens = random.randint(50, 150)
    elif complexity == "medium":
        input_tokens = random.randint(300, 500)
        output_tokens = random.randint(150, 300)
    else:
        input_tokens = random.randint(500, 800)
        output_tokens = random.randint(300, 500)
    
    queries.append({
        "complexity": complexity,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens
    })


In [10]:

# Run all three strategies
for strategy in ["always_gpt4", "always_llama", "smart_routing"]:
    result = simulate_routing_cost(queries, strategy)
    print(f"Strategy: {result['strategy']}")
    print(f"  Total cost: ${result['total_cost_usd']}")
    print(f"  Avg per query: ${result['avg_cost_per_query']}")
    print(f"  Model usage: {result['model_usage']}")
    print()

Strategy: always_gpt4
  Total cost: $19.10187
  Avg per query: $0.019102
  Model usage: {'gpt-4': 1000}

Strategy: always_llama
  Total cost: $0.02852
  Avg per query: $2.9e-05
  Model usage: {'llama-3.1-8b': 1000}

Strategy: smart_routing
  Total cost: $4.760279
  Avg per query: $0.00476
  Model usage: {'gpt-4': 107, 'llama-3.1-8b': 630, 'mixtral-8x7b': 263}



In [11]:
def calculate_savings_report(queries: list) -> dict:
    """
    this is what gets logged to MLflow after every batch
    gives recruiters and stakeholders a live cost dashboard
    """
    baseline = simulate_routing_cost(queries, "always_gpt4")
    smart = simulate_routing_cost(queries, "smart_routing")
    
    savings_usd = baseline["total_cost_usd"] - smart["total_cost_usd"]
    savings_pct = (savings_usd / baseline["total_cost_usd"]) * 100
    
    return {
        "baseline_cost": baseline["total_cost_usd"],
        "smart_routing_cost": smart["total_cost_usd"],
        "savings_usd": round(savings_usd, 4),
        "savings_pct": round(savings_pct, 2),
        "queries_analyzed": len(queries),
        "model_distribution": smart["model_usage"]
    }

report = calculate_savings_report(queries)
print("=== LLM Router Cost Report ===")
print(f"Baseline (GPT-4 only):    ${report['baseline_cost']}")
print(f"Smart routing:            ${report['smart_routing_cost']}")
print(f"Savings:                  ${report['savings_usd']} ({report['savings_pct']}%)")
print(f"Queries analyzed:         {report['queries_analyzed']}")
print(f"Model distribution:       {report['model_distribution']}")

=== LLM Router Cost Report ===
Baseline (GPT-4 only):    $19.10187
Smart routing:            $4.760279
Savings:                  $14.3416 (75.08%)
Queries analyzed:         1000
Model distribution:       {'gpt-4': 107, 'llama-3.1-8b': 630, 'mixtral-8x7b': 263}


In [12]:
def log_real_query_cost(groq_response: dict, model: str) -> dict:
    """
    WHY: simulate cost from REAL token counts returned by Groq API
    not estimated — actual tokens used per query
    This is what gets logged to MLflow in Project 4
    """
    usage = groq_response["usage"]
    
    input_tokens = usage["prompt_tokens"]
    output_tokens = usage["completion_tokens"]
    total_tokens = usage["total_tokens"]
    
    cost = calculate_cost(model, input_tokens, output_tokens)
    
    # Simulate what GPT-4 would have cost for same query
    gpt4_cost = calculate_cost("gpt-4", input_tokens, output_tokens)
    
    saved = gpt4_cost["total_cost_usd"] - cost["total_cost_usd"]
    
    return {
        "model_used": model,
        "prompt_tokens": input_tokens,
        "completion_tokens": output_tokens,
        "total_tokens": total_tokens,
        "actual_cost_usd": cost["total_cost_usd"],
        "gpt4_equivalent_cost_usd": gpt4_cost["total_cost_usd"],
        "saved_usd": round(saved, 6),
        "groq_time_seconds": usage["total_time"]
    }

In [14]:
# Use real response data from yesterday's Day 58 notebook
# These are actual token counts from your Groq API calls
real_responses = [
    {"usage": {"prompt_tokens": 216, "completion_tokens": 99, 
               "total_tokens": 315, "total_time": 0.339}},
    {"usage": {"prompt_tokens": 162, "completion_tokens": 169,
               "total_tokens": 331, "total_time": 0.281}},
    {"usage": {"prompt_tokens": 176, "completion_tokens": 235,
               "total_tokens": 411, "total_time": 0.499}}
]

print("=== Real Query Cost Log ===")
total_actual = 0
total_gpt4 = 0

for i, response in enumerate(real_responses):
    log = log_real_query_cost(response, "llama-3.1-8b")
    total_actual += log["actual_cost_usd"]
    total_gpt4 += log["gpt4_equivalent_cost_usd"]
    print(f"Query {i+1}: {log['total_tokens']} tokens | "
          f"Actual: ${log['actual_cost_usd']} | "
          f"GPT-4 would cost: ${log['gpt4_equivalent_cost_usd']} | "
          f"Saved: ${log['saved_usd']} | "
          f"Time: {log['groq_time_seconds']}s")

print(f"\nTotal actual cost: ${round(total_actual, 6)}")
print(f"Total if GPT-4:    ${round(total_gpt4, 6)}")
print(f"Total saved:       ${round(total_gpt4 - total_actual, 6)}")

=== Real Query Cost Log ===
Query 1: 315 tokens | Actual: $1.9e-05 | GPT-4 would cost: $0.01242 | Saved: $0.012401 | Time: 0.339s
Query 2: 331 tokens | Actual: $2.2e-05 | GPT-4 would cost: $0.015 | Saved: $0.014978 | Time: 0.281s
Query 3: 411 tokens | Actual: $2.8e-05 | GPT-4 would cost: $0.01938 | Saved: $0.019352 | Time: 0.499s

Total actual cost: $6.9e-05
Total if GPT-4:    $0.0468
Total saved:       $0.046731


## Day 60 Cost Mathematics and Token Pricing

### What a token is
~4 characters, ~0.75 words. Subword units, not whole words.
Vocabulary: 50,000-100,000 tokens covers any text by combining pieces.

### Pricing reality (per 1M tokens)
GPT-4:        $30 input / $60 output
GPT-3.5:      $0.50 input / $1.50 output  
Mixtral 8x7B: $0.24 input / $0.24 output
Llama 3.1 8B: $0.05 input / $0.08 output

### Key numbers
- 1000 queries/day always_gpt4: $19.10
- 1000 queries/day smart_routing: $4.76
- Savings: 75.08%
- At 10K queries/day: $4,302/month saved

### What goes into Project 4 MLflow
Every query logs: model used, tokens, actual cost, GPT-4 equivalent, saved, latency
Recruiters see live cost dashboard — not simulated, grounded in real token counts

### Critical insight
10% complex queries on GPT-4 dominate total cost.
Routing strategy must minimize GPT-4 calls while maintaining quality.
That's the entire business case for the LLM router.